# AI-Impact Essay Analysis Toolkit: 02 ML (Metric Learning)

## Theme

The **impact of generative AI on writing**, specifically how student essays may have shifted as these tools became widely available.

### Context

Because many classes have relied on **similar essay prompts** both before and after the introduction of generative AI, we have a consistent baseline for comparison.

### Idea

The central aim is to investigate how generative AI has **influenced student writing** and to determine whether those effects can be formally studied.

## Strategy

The approach is to design **metrics** that can reliably separate essays written before generative AI from those written afterwards.

### Caveat

Any metrics developed must remain **stable and dependable**, ensuring they do not produce misleading differences.

## Preliminary Ideas

+ **Signature phrases**: Spotting distinctive word choices that recur.  
+ **Structural features**: Examining recognisable sentence or paragraph patterns.  
+ **Punctuation**: Identifying habits in marks and symbols.  
  + **More advanced methods**: Eventually moving towards deeper stylistic analysis.

## Theme

The **impact of generative AI on writing**, specifically how student essays may have shifted as these tools became widely available.

### Context

Because many classes have relied on **similar essay prompts** both before and after the introduction of generative AI, we have a consistent baseline for comparison.

### Idea

The central aim is to investigate how generative AI has **influenced student writing** and to determine whether those effects can be formally studied.

## Strategy

The approach is to design **metrics** that can reliably separate essays written before generative AI from those written afterwards.

### Caveat

Any metrics developed must remain **stable and dependable**, ensuring they do not produce misleading differences.

## Preliminary Ideas

+ **Signature phrases**: Spotting distinctive word choices that recur.  
+ **Structural features**: Examining recognisable sentence or paragraph patterns.
+ **Punctuation**: Identifying habits in marks and symbols.  
  + **More advanced methods**: Eventually moving towards deeper stylistic analysis.

## Dependencies

Locally, just:

```python
# /// script
# requires-python = '>=3.12'
# dependencies = [
#   'nltk>=3.8',
#   'spacy>=3.7',
#   'numpy>=1.26',
#   'pandas>=2.1',
#   'matplotlib>=3.8',
#   'seaborn>=0.13',
#   'scikit-learn>=1.4',
#   'scipy>=1.12',
#   'umap-learn>=0.5',
#   'sentence-transformers>=2.7',
#   'shap>=0.45',
#   'pdfplumber>=0.10',
#   'fpdf2>=2.7',
# ]
# ///

# First run only — download required NLTK + spaCy assets:
# python -m nltk.downloader punkt averaged_perceptron_tagger stopwords
# python -m spacy download en_core_web_sm
```

In [ ]:
import sys

# 1. Install packages
# Using `%pip` instead of `!pip` is best practice in notebooks as it ensures the packages are installed in the specific environment the notebook is currently using.
# Install the necessary NLP and ML libraries, ensuring minimum versions are met for compatibility.
%pip install nltk>=3.8 spacy>=3.7 numpy>=1.26 pandas>=2.1 matplotlib>=3.8 seaborn>=0.13 scikit-learn>=1.4 scipy>=1.12 umap-learn>=0.5 sentence-transformers>=2.7 shap>=0.45

# 2. Download NLTK data
import nltk
# Iterate through and fetch specific NLTK resources required for tokenisation and part-of-speech tagging.
for resource in ['punkt', 'averaged_perceptron_tagger', 'stopwords', 'punkt_tab']:
    nltk.download(resource)

# 3. Download spaCy model
# This runs the shell command only if the model isn't already available
try:
    # Attempt to load the small English model directly to verify existence.
    import en_core_web_sm
except ImportError:
    # Use the current python executable to download the model if the import fails.
    !{sys.executable} -m spacy download en_core_web_sm

In [ ]:
# Deps for data ingestion

# NOTE: pdfplumber is the extraction workhorse. It handles real-world
# academic PDFs (multi-column, headers/footers, embedded fonts) more
# reliably than PyPDF2 or pypdf for prose text.
# fpdf2 is only imported in demo mode to generate synthetic PDFs.
%pip install -q pdfplumber>=0.10 fpdf2>=2.7

## Dummy Data

Generated with Gemini for prototyping.


In [ ]:
text1 = '''The intersection of urban planning and the "psychogeography" of 21st-century digital nomads presents a fascinating, if somewhat chaotic, case study in modern sociology. When we look at how people actually inhabit spaces—not just the blueprints drafted in some glass-walled office in Canary Wharf—there is a palpable sense of friction. You see, the modern worker isn't tethered to a desk; they’re tethered to a Wi-Fi signal. This creates a weird sort of "liminal existence" where the physical environment becomes secondary to the digital one. I was chatting with a bloke in a coffee shop last week—totally informal setting, right?—and he mentioned how he feels more at home in a pixelated Discord server than in his own flat in Shoreditch. It’s a bit mental when you think about it.
The aesthetic of these spaces often tries to bridge the gap. We see "industrial chic" everywhere: exposed brick, Edison bulbs, and those uncomfortable metal chairs that seem designed to discourage anyone from sitting longer than it takes to drink a flat white. Is this a deliberate design choice to increase turnover, or just a lack of imagination? Probably both. Interestingly, the American influence on these spaces is undeniable. Even in the heart of London, you’ll find menus listing "sidewalk seating" instead of pavements, and the coffee sizes are getting out of hand. It’s a linguistic colonisation that happens almost by osmosis. There’s a certain labour involved in maintaining a local identity against the crushing weight of globalized "Instagrammable" decor.
Yet, we must also consider the economic underpinnings. Real estate developers aren't altruists; they are seeking to maximize yield per square foot. (Sometimes I wonder if they even know what a square foot feels like, or if it’s just a number on a spreadsheet). In the rush to "disrupt" the housing market, they’ve created these co-living spaces that are basically dorms for adults who earn six figures. It’s a strange irony. You pay a premium to live like a student again, all for the sake of "community" that usually amounts to a shared fridge and a Slack channel no one checks.
There are errors in the way we perceive these shifts, too. We often conflate proximity with intimacy. Just because you’re breathing the same recycled air as twenty other "creatives" doesn't mean you’re collaborating. Often, the silence in these rooms is deafening, broken only by the clacking of mechanical keyboards. It’s a performative kind of work. We’re all actors in a play called "Being Productive," and the set design is a repurposed warehouse. If we don't start prioritizing actual human connection over digital throughput, we’re going to find ourselves very lonely in very beautiful rooms. The data suggests a rise in "urban loneliness," which is a fancy way of saying we’re surrounded by people but haven't spoken to a soul all day. It’s high time we re-evaluating the metrics of a successful city—it shouldn't just be about GDP or high-speed rail, but about whether or not you can find a decent place to sit and talk without having to buy a £5 latte first.'''

text2 = '''The transition toward renewable energy sources is not merely a technical challenge but a fundamental shift in the global geopolitical landscape. For decades, the world has been dominated by the "petro-state" model, where power was concentrated in the hands of those with vast oil reserves. However, as we move toward a decentralized grid powered by solar, wind, and hydroelectricity, the old alliances are beginning to crumble. This transformation is often described as "The Great Greening," though the reality is far more complex and significantly more fraught with peril than the catchy title suggests.
One of the primary drivers of this change is the plummeting cost of photovoltaic cells. Efficiency has increased tenfold over the last twenty years, making solar power the cheapest form of electricity in history in many parts of the world. This is a remarkable feat of engineering and economies of scale. Nevertheless, the intermittent nature of renewable energy remains a significant hurdle. We cannot simply flip a switch and expect the sun to shine at midnight; therefore, the development of long-term battery storage is the "Holy Grail" of the energy sector. Without it, we are merely trading one form of dependency for another—specifically, a dependency on the rare earth minerals required for battery production.
Furthermore, the social implications of this transition are often overlooked in the hallowed halls of policy-making. What happens to the coal miner in West Virginia or the rig worker in the North Sea? These are individuals whose identities and livelihoods are inextricably linked to the old energy regime. To tell them to "learn to code" is not only dismissive but fundamentally cruel. A "Just Transition" must involve more than just subsidies; it requires a genuine reinvestment in the fabric of these communities. We must ensure that the green economy is an inclusive one, rather than a playground for the technological elite.
In addition to the social and economic factors, we must address the environmental impact of the green transition itself. It is a bit of a paradox, isn't it? To save the planet, we must dig massive holes in the ground to extract lithium and cobalt. This mining often takes place in countries with poor human rights records and lax environmental regulations. We are, in effect, exporting our ecological footprint to the Global South. If we are not careful, the "Green Revolution" will be built on the same foundations of exploitation as the Industrial Revolution that preceded it.
Ultimately, the path forward requires a multi-faceted approach that balances innovation with equity. We need better tech, yes, but we also need better politics. We must move beyond the binary of "pro-oil" or "anti-growth" and find a middle ground that prioritizes the long-term health of our species. The clock is ticking, and the atmosphere doesn't care about our political bickering. It only responds to the parts-per-million of carbon. We have the tools to change the trajectory, but do we have the collective will? That is the question that will define the next century of human history, and currently, the answer remains frustratingly elusive.'''

## Configuration for Data Ingestion

This is the only part of the config that a user might need to make changes to.

In [ ]:
# =============================================================================
# DATA INGESTION CONFIGURATION
# =============================================================================

# -- Paths --------------------------------------------------------------------
DATA_ROOT = '/data'   # parent folder containing the YYYYMM-... dirs

# -- Courses ------------------------------------------------------------------
# Add one entry per course. The string is matched as a substring of the folder
# name (case-insensitive), so partial matches are fine.
# Format: (course_id_substring, human_readable_label)
COURSES: list[tuple[str, str]] = [
  ('CS6460', 'Educational Technology'),
  # ('CHEM612', 'Designing Organic Synthesis'),   # <-- add more courses like this
]

# -- Assignments to include ---------------------------------------------------
# Substrings matched case-insensitively against assignment folder names.
# Leave empty ( [] ) to include ALL assignments found.
ASSIGNMENTS: list[str] = [
  'Assignment_1',
  'Assignment_2',
  'Assignment_3',
  'Project_Paper',
  'Qualifier_Question',
  # 'Assignment_4',   # uncomment if needed
]

# -- Temporal split -----------------------------------------------------------
# Essays from semesters whose YYYYMM prefix is <= this value are labelled
# 'Pre-ChatGPT'; strictly greater -> 'Post-ChatGPT'.
# ChatGPT public release: November 2022 -> 202211.
# We use 202208 (FA22) as the last clean pre-AI semester.
CHATGPT_CUTOFF_YYYYMM: int = 202208

# -- Demo mode ----------------------------------------------------------------
# When True, real data are ignored and a small synthetic corpus is generated.
# Flip to False (or set env var DEMO_MODE=0) when real data are available.
import os
DEMO_MODE: bool = bool(int(os.environ.get('DEMO_MODE', '1')))

# -- Misc ---------------------------------------------------------------------
# Maximum characters read per PDF page (guards against runaway extraction).
MAX_CHARS_PER_PAGE: int = 50_000

### Imports for Ingestion

In [ ]:
# NOTE: pdfplumber is the extraction workhorse. It handles real-world
# academic PDFs (multi-column, headers/footers, embedded fonts) more
# reliably than PyPDF2 or pypdf for prose text.
# fpdf2 is only imported in demo mode to generate synthetic PDFs.
%pip install -q pdfplumber>=0.10 fpdf2>=2.7

import re
import random
import textwrap
import tempfile
import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterator

import pandas as pd
import pdfplumber

### Data Model

In [ ]:
@dataclass(frozen=True)
class Semester:
  '''Parsed representation of a semester folder name.'''
  yyyymm:    int     # e.g. 202208
  code:      str     # e.g. 'FA22'
  canvas_id: str     # e.g. '123456'
  course:    str     # human-readable label from COURSES config
  path:      Path

  @property
  def era(self) -> str:
    return 'Pre-ChatGPT' if self.yyyymm <= CHATGPT_CUTOFF_YYYYMM else 'Post-ChatGPT'

  @property
  def label(self) -> str:
    '''Short label for plots, e.g. "FA22".'''
    return self.code


@dataclass
class EssayRecord:
  '''One student submission.'''
  semester:    Semester
  assignment:  str      # normalised assignment name, e.g. 'Project_Paper'
  student_id:  str      # numeric prefix of student folder
  student_name: str     # remainder of student folder name
  pdf_path:    Path
  text:        str = field(default='', repr=False)

  @property
  def uid(self) -> str:
    '''Stable unique ID: canvas_id + assignment + student_id.'''
    return f'{self.semester.canvas_id}__{self.assignment}__{self.student_id}'

## Folder Parsing

To use this script with real data, ensure `DEMO_MODE` is set to `0` (or `False`) in the configuration section above.

The script expects a specific hierarchical directory structure within the `DATA_ROOT`:

```text
DATA_ROOT/
'-- 202208-CS6460_Educational_Technology_(FA22)-987654/       # Semester Folder
    '-- submissions/                                          # Static Subdirectory
        '-- 900000_Assignment_1/                              # Assignment Folder
            '-- 88001_Student_Name/                           # Student Folder
                '-- submission.pdf                            # The Essay PDF
```

*   **Semester Folders**: Must match the regex `YYYYMM-CourseName_(SSYY)-CanvasID`.
*   **Assignment Folders**: Should contain the substring defined in the `ASSIGNMENTS` config list.
*   **Student Folders**: Must start with a numeric ID followed by an underscore.

In [ ]:
# Matches: YYYYMM-CourseName_(SSYY)-CanvasID
# e.g.   : 202208-CS6460_Educational_Technology_(FA22)-987654
_FOLDER_RE = re.compile(
  r'^(?P<yyyymm>\d{6})-(?P<course_slug>.+?)_\((?P<code>[A-Z]{2}\d{2})\)-(?P<canvas_id>\d+)$'
)


def _match_course(course_slug: str) -> str | None:
  '''
  Return the human-readable course label if *course_slug* matches any entry
  in COURSES, else None.
  '''
  slug_lower = course_slug.lower()
  for substring, label in COURSES:
    if substring.lower() in slug_lower:
      return label
  return None


def _match_assignment(folder_name: str) -> str | None:
  '''
  Return the normalised assignment name if the folder matches any entry in
  ASSIGNMENTS (or if ASSIGNMENTS is empty, return the folder name as-is).
  '''
  name_lower = folder_name.lower()

  if not ASSIGNMENTS:
    # Strip the numeric prefix and return the remainder
    _, _, rest = folder_name.partition('_')
    return rest or folder_name

  for assignment in ASSIGNMENTS:
    if assignment.lower() in name_lower:
      return assignment   # use the configured canonical name

  return None


def discover_semesters(data_root: Path) -> list[Semester]:
  '''
  Walk *data_root* and return one Semester for every folder that matches
  the naming convention AND at least one configured course substring.
  Results are sorted chronologically (by yyyymm, then canvas_id).
  '''
  semesters: list[Semester] = []

  for entry in sorted(data_root.iterdir()):
    if not entry.is_dir():
      continue
    m = _FOLDER_RE.match(entry.name)
    if not m:
      continue
    course_label = _match_course(m.group('course_slug'))
    if course_label is None:
      continue
    semesters.append(Semester(
      yyyymm    = int(m.group('yyyymm')),
      code      = m.group('code'),
      canvas_id = m.group('canvas_id'),
      course    = course_label,
      path      = entry,
    ))

  return sorted(semesters, key=lambda s: (s.yyyymm, s.canvas_id))


def iter_essay_records(semesters: list[Semester]) -> Iterator[EssayRecord]:
  '''
  Yield one EssayRecord per PDF found under
  semester/submissions/<assignment>/<student>/<file>.pdf.

  Skips any folder/file that does not conform to the expected layout
  and emits a warning so problems are visible without aborting the run.
  '''
  sub_re     = re.compile(r'^\d+_')   # numeric prefix check

  for sem in semesters:
    submissions_dir = sem.path / 'submissions'
    if not submissions_dir.is_dir():
      print(f'  [warn] no submissions/ folder in {sem.path.name}')
      continue

    for assign_dir in sorted(submissions_dir.iterdir()):
      if not assign_dir.is_dir():
        continue
      assignment = _match_assignment(assign_dir.name)
      if assignment is None:
        continue

      for student_dir in sorted(assign_dir.iterdir()):
        if not student_dir.is_dir() or not sub_re.match(student_dir.name):
          continue

        student_id, _, student_name = student_dir.name.partition('_')

        pdfs = list(student_dir.glob('*.pdf'))
        if not pdfs:
          print(f'  [warn] no PDF in {student_dir}')
          continue
        if len(pdfs) > 1:
          print(f'  [warn] multiple PDFs in {student_dir}; using first alphabetically')

        yield EssayRecord(
          semester     = sem,
          assignment   = assignment,
          student_id   = student_id,
          student_name = student_name,
          pdf_path     = sorted(pdfs)[0],
        )

### PDF Extraction

In [ ]:
def extract_text_from_pdf(path: Path, *, max_chars: int = MAX_CHARS_PER_PAGE) -> str:
  '''
  Extract plain text from *path* using pdfplumber.

  Strategy
  --------
  - Extract each page's text separately to apply per-page char cap (guards
    against bibliography/appendix bloat).
  - Collapse runs of blank lines to paragraph breaks so the downstream
    split_paragraphs() function works correctly.
  - Returns an empty string on failure rather than raising, so one bad PDF
    does not abort ingestion.
  '''
  try:
    pages: list[str] = []
    with pdfplumber.open(path) as pdf:
      for page in pdf.pages:
        raw = page.extract_text(x_tolerance=2, y_tolerance=2) or ''
        pages.append(raw[:max_chars])

    full = '\n\n'.join(pages)

    # Normalise whitespace: collapse 3+ blank lines to exactly two
    full = re.sub(r'\n{3,}', '\n\n', full)
    return full.strip()

  except Exception as exc:   # noqa: BLE001
    print(f'  [warn] could not extract {path.name}: {exc}')
    return ''


def load_records(records: list[EssayRecord]) -> list[EssayRecord]:
  '''
  Populate the .text field for each record in-place.
  Records with empty text after extraction are dropped with a warning.
  '''
  loaded: list[EssayRecord] = []
  for rec in records:
    text = extract_text_from_pdf(rec.pdf_path)
    if not text:
      print(f'  [warn] empty text for {rec.uid}; skipping')
      continue
    # dataclass is frozen at the Semester level but EssayRecord is not
    rec.text = text
    loaded.append(rec)
  return loaded

## Demo Mode (Synthetic Data)

In [ ]:
import re
import random
import textwrap
import tempfile
import hashlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterator

import pandas as pd
import pdfplumber

# ---------------------------------------------------------------------------
# Synthetic data generator.
# Produces realistic folder + PDF structure so both notebooks can be
# developed and tested without touching real student data.
# ---------------------------------------------------------------------------

_PRE_AI_TEMPLATE = textwrap.dedent('''\
  {topic} is an important subject in educational technology.
  There are several reasons why this matters. First, {reason1}.
  Second, {reason2}. I think this shows that {conclusion}.

  Robespierre is an interesting figure because he seemed to believe in
  what he was doing. Whether that was a good thing is still debated.

  In the end, {final}.
''')

_POST_AI_TEMPLATE = textwrap.dedent('''\
  {topic} represents a pivotal and multifaceted dimension of educational
  technology. It is important to note that this transformative landscape
  encompasses several crucial aspects. Furthermore, {reason1}.

  Moreover, {reason2}. Navigating the complex intersection of theory and
  practice, scholars have fostered a comprehensive understanding of the
  paradigm. Additionally, this robust framework underscores the vibrant
  ecosystem of modern pedagogy.

  In conclusion, {topic} is not merely a concept but a testament to the
  broader human endeavour for knowledge. Its intricate tapestry of causes
  and consequences continues to illuminate our understanding of {final}.
''')

_FILLERS = {
  'topic':      ['personalised learning', 'AI in classrooms', 'peer assessment',
                 'gamification', 'adaptive systems'],
  'reason1':    ['students learn better with feedback', 'technology reduces barriers',
                 'engagement metrics improve outcomes'],
  'reason2':    ['the evidence supports early intervention', 'costs decrease over time',
                 'teachers report higher satisfaction'],
  'conclusion': ['this field is worth further study', 'policy should reflect these findings',
                 'more research is needed'],
  'final':      ['education will continue to evolve', 'the future remains uncertain',
                 'collaboration is key'],
}


def _random_fill(template: str, rng: random.Random) -> str:
  return template.format(**{k: rng.choice(v) for k, v in _FILLERS.items()})


def _write_synthetic_pdf(text: str, path: Path) -> None:
  '''Write *text* to a minimal single-page PDF at *path* using fpdf2.'''
  from fpdf import FPDF   # imported here so fpdf2 is only required in demo mode
  pdf = FPDF()
  pdf.add_page()
  pdf.set_font('Helvetica', size=6)
  # Set a generous margin and automatic page breaking
  pdf.set_auto_page_break(auto=True, margin=15)

  # We use a smaller line height (h=3) to ensure lines fit vertically,
  # and set the width to 0 (full page width) for multi_cell.
  for line in text.splitlines():
    if line.strip():
        pdf.multi_cell(w=0, h=3, text=line)
        pdf.ln(2)

  pdf.output(str(path))


def build_demo_corpus(
  root:          Path,
  *,
  n_students:    int  = 3,
  random_seed:   int  = 42,
) -> None:
  '''
  Construct a minimal but structurally faithful synthetic dataset under *root*.

  Semesters generated: SP21, SU21, FA21 (pre) + SP22, FA22 (pre) + SP23, FA23 (post).
  Assignments generated: those listed in ASSIGNMENTS (or a default set if empty).
  '''
  rng = random.Random(random_seed)

  fake_semesters = [
    ('202102', 'SP21', '100001'),
    ('202105', 'SU21', '100002'),
    ('202108', 'FA21', '100003'),
    ('202202', 'SP22', '100004'),
    ('202208', 'FA22', '100005'),   # last pre-AI semester
    ('202302', 'SP23', '100006'),
    ('202308', 'FA23', '100007'),
  ]

  assignments_to_make = ASSIGNMENTS if ASSIGNMENTS else [
    'Assignment_1', 'Assignment_2', 'Project_Paper',
  ]

  course_slug = COURSES[0][0] if COURSES else 'CS6460'

  fake_students = [
    ('88001', 'Fatima_Khan'),
    ('88002', 'Misbah_Nazir'),
    ('88003', 'Qurratulain_Sultan'),
    ('88004', 'Sawaira_Jutt'),
    ('88005', 'Umm_Hafsa'),
  ][:n_students]

  for yyyymm, code, canvas_id in fake_semesters:
    era_cutoff = int(yyyymm) <= CHATGPT_CUTOFF_YYYYMM
    template   = _PRE_AI_TEMPLATE if era_cutoff else _POST_AI_TEMPLATE

    sem_dir = root / f'{yyyymm}-{course_slug}_Educational_Technology_({code})-{canvas_id}'

    for i, assignment in enumerate(assignments_to_make):
      assign_prefix = str(900000 + i)
      assign_dir    = sem_dir / 'submissions' / f'{assign_prefix}_{assignment}'

      for sid, sname in fake_students:
        student_dir = assign_dir / f'{sid}_{sname}'
        student_dir.mkdir(parents=True, exist_ok=True)
        pdf_path = student_dir / 'submission.pdf'
        if not pdf_path.exists():
          body = _random_fill(template, rng)
          # Vary length slightly so metrics are not identical across students
          body = body + (' ' + rng.choice(list(_FILLERS['conclusion']))) * rng.randint(0, 3)
          _write_synthetic_pdf(body, pdf_path)

  print(f'[demo] Synthetic dataset written to: {root}')

## Corpus Assembly

In [ ]:
def assemble_corpus(
  records: list[EssayRecord],
  *,
  group_by: str = 'era',         # 'era' | 'semester' | 'assignment'
) -> 'LabelledCorpus':
  '''
  Convert a flat list of EssayRecords into a LabelledCorpus dict.

  group_by options
  ----------------
  'era'        : two groups -- 'Pre-ChatGPT' / 'Post-ChatGPT'
  'semester'   : one group per semester code (e.g. 'FA21', 'SP23')
  'assignment' : one group per assignment name
  '''
  corpus: dict[str, list[str]] = {}

  for rec in records:
    match group_by:
      case 'era':
        key = rec.semester.era
      case 'semester':
        key = rec.semester.label
      case 'assignment':
        key = rec.assignment
      case _:
        raise ValueError(f"group_by must be 'era', 'semester', or 'assignment'; got '{group_by}'")

    corpus.setdefault(key, []).append(rec.text)

  return corpus


def summarise_corpus(records: list[EssayRecord]) -> pd.DataFrame:
  '''
  Return a human-readable summary table of what was loaded --
  useful as a quick sanity check before running either notebook's pipeline.
  '''
  rows = [
    {
      'semester':   rec.semester.code,
      'yyyymm':     rec.semester.yyyymm,
      'era':        rec.semester.era,
      'assignment': rec.assignment,
      'student_id': rec.student_id,
      'word_count': len(rec.text.split()),
      'uid':        rec.uid,
    }
    for rec in records
  ]
  df = pd.DataFrame(rows)
  if df.empty:
    return df
  print(df.groupby(['era', 'semester', 'assignment']).size()
          .rename('n_essays')
          .to_string())
  return df

## Ingestion Entry Point

In [ ]:
# ---------------------------------------------------------------------------
# This is the only cell that actually *runs* the ingestion.
# Both notebooks import the corpus from `_RECORDS` and `CORPUS`.
# ---------------------------------------------------------------------------

if DEMO_MODE:
  print('[demo] Demo mode ON -- generating synthetic data.')
  _DEMO_ROOT = Path(tempfile.mkdtemp(prefix='essay_demo_'))
  build_demo_corpus(_DEMO_ROOT)
  _data_root = _DEMO_ROOT
else:
  _data_root = Path(DATA_ROOT)
  print(f'[info] Loading from: {_data_root}')

_SEMESTERS = discover_semesters(_data_root)
print(f'[info] Found {len(_SEMESTERS)} matching semester folder(s).')

_RAW_RECORDS  = list(iter_essay_records(_SEMESTERS))
print(f'[info] Discovered {len(_RAW_RECORDS)} submission(s).')

_RECORDS      = load_records(_RAW_RECORDS)
print(f'[info] Loaded {len(_RECORDS)} submission(s) with non-empty text.')

_SUMMARY_DF   = summarise_corpus(_RECORDS)

# Default grouping used by both notebooks
CORPUS: 'LabelledCorpus' = assemble_corpus(_RECORDS, group_by='era')
print(f'\n[info] Corpus groups: { {k: len(v) for k, v in CORPUS.items()} }')

## Imports & Global Config

This section initialises the environment, configuring the visual aesthetic using the Agrarian palette and loading pre-trained transformers.

### Theoretical Context
- **spaCy (`en_core_web_sm`)**: Utilises a Multi-Hash Embed CNN pipeline for tokenisation and POS tagging. It maps tokens to dense vectors using Bloom embeddings ([Honnibal & Montani, 2017](https://spacy.io)).
- **Sentence-Transformers**: Implements Siamese BERT networks. The `all-MiniLM-L6-v2` model compresses knowledge via Distillation, mapping sentences to a hypersphere where cosine similarity $\cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$ represents semantic proximity ([Reimers & Gurevych, 2019](https://arxiv.org/abs/1908.10084)).

In [ ]:
from __future__ import annotations

import re
import string
import warnings
from dataclasses import dataclass, field
from typing import Any, Protocol

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import spacy
import umap
from nltk.tokenize import sent_tokenize, word_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Suppress FutureWarnings to maintain a clean output during the analysis.
warnings.filterwarnings('ignore', category=FutureWarning)

# ---------------------------------------------------------------------------
# Agrarian Palette
# ---------------------------------------------------------------------------
# Define a primary dictionary for the Agrarian visual identity.
UAF = {
  'blue':   '#403F84',
  'sky':    '#2F9FD9',
  'yellow': '#F8D727',
  'green':  '#005C45',
}

# Accessible sequential / diverging variants derived from the brand palette.
# Used consistently throughout all visualisations.
_PALETTE_TWO   = [UAF['sky'], UAF['yellow']]
_PALETTE_FOUR  = [UAF['blue'], UAF['sky'], UAF['yellow'], UAF['green']]
_CMAP_DIV      = 'RdBu_r'   # good diverging default; colourblind-safe
_CMAP_SEQ      = 'Blues'

# Apply the Agrarian theme to Seaborn and Matplotlib global parameters.
sns.set_theme(style='whitegrid', palette=_PALETTE_FOUR)
plt.rcParams.update({
  'figure.dpi':       130,
  'font.size':        11,
  'axes.titlesize':   12,
  'axes.labelsize':   11,
  'legend.fontsize':  10,
  'figure.facecolor': 'white',
})

# Load the spaCy model for syntactic analysis and the SBERT model for semantic embeddings.
_NLP    = spacy.load('en_core_web_sm')
_SBERT  = SentenceTransformer('all-MiniLM-L6-v2')  # ~90 MB, free, offline after first run

## Shared Types & Protocols

We utilise Python's `typing.Protocol` to implement structural subtyping (PEP 544). This allows for 'duck-typed' feature extractors, ensuring the system is extensible without rigid inheritance hierarchies.

### Theoretical Context
- **Interface Segregation**: By defining a `FeatureExtractor` protocol, we ensure that any class providing an `extract` method satisfies the pipeline contract.
- **Assumptions**: We assume that any metric can be represented as a scalar float value, which allows for consistent matrix construction downstream.

In [ ]:
# ---------------------------------------------------------------------------
# Protocols define the contracts that all feature extractors must satisfy.
# Using Protocol rather than ABC keeps things structurally typed (duck-typed)
# which is more Pythonic and easier to extend.
# ---------------------------------------------------------------------------

type LabelledCorpus = dict[str, list[str]]
'''Mapping of group label -> list of raw essay strings.'''

type FeatureMatrix = pd.DataFrame
'''Rows = essays, columns = feature names, values = floats.'''


class FeatureExtractor(Protocol):
  '''
  Anything that can turn a raw essay string into a dict of named floats
  qualifies as a FeatureExtractor - no inheritance required.
  '''
  name: str

  def extract(self, text: str) -> dict[str, float]:
    ...


@dataclass(frozen=True)
class DiscoveredMetric:
  '''A metric surfaced automatically by the ML pipeline.'''
  name:        str
  importance:  float          # relative feature importance [0, 1]
  direction:   str            # 'higher_in_post' | 'higher_in_pre' | 'unclear'
  description: str
  example_values: dict[str, float] = field(default_factory=dict)
  # e.g. {'Pre-ChatGPT': 0.23, 'Post-ChatGPT': 0.61}

## Text Preprocessing Utilities

Raw text is normalised and vectorised to prepare for statistical analysis.

### Methodology
- **Tokenisation**: Word-level segmentation is performed via NLTK's `punkt` algorithm, an unsupervised machine learning model for boundary detection.
- **Aggregation**: Document embeddings are calculated as the centroid of sentence vectors: $\mathbf{V}_{doc} = \frac{1}{n} \sum_{i=1}^{n} \mathbf{v}_i$. This assumes semantic stability across the document's progression.

In [ ]:
# Reused verbatim from 01 Expert Metrics where possible;
# extended with sentence-embedding helpers.

def split_paragraphs(text: str) -> list[str]:
  '''Split on one or more blank lines; discard empty paragraphs.'''
  # Identify paragraph boundaries using double newline characters.
  return [p.strip() for p in re.split(r'\n{2,}', text) if p.strip()]


def tokenise_words(text: str, *, lowercase: bool = True) -> list[str]:
  '''Word tokens stripped of punctuation; optionally lowercased.'''
  # Standardise the case of the text if required.
  tokens = word_tokenize(text.lower() if lowercase else text)
  # Filter out tokens that consist solely of punctuation marks.
  return [t for t in tokens if t not in string.punctuation]


def embed_sentences(sentences: list[str]) -> np.ndarray:
  '''
  Return an (n, 384)-shaped float32 array of SBERT sentence embeddings.
  Using SBERT rather than TF-IDF here gives genuine semantic similarity
  rather than lexical overlap, which matters for redundancy detection.
  '''
  # Check for empty input to avoid processing errors.
  if not sentences:
    return np.zeros((0, 384), dtype=np.float32)
  # Generate dense vector representations for each sentence.
  return _SBERT.encode(sentences, show_progress_bar=False, normalize_embeddings=True)


def embed_text(text: str) -> np.ndarray:
  '''Single-document embedding (mean of sentence embeddings).'''
  # Break the document into individual sentences.
  sents = sent_tokenize(text)
  # Compute the embeddings and return the average vector as the document fingerprint.
  vecs  = embed_sentences(sents)
  return vecs.mean(axis=0) if len(vecs) else np.zeros(384, dtype=np.float32)

## Feature Extractors

This module implements five specialised extractors to capture linguistic 'fingerprints'.

### Analytical Framework
- **Lexical Diversity**: Uses the Moving-Average Type-Token Ratio (MATTR) to control for document length bias. Unlike raw TTR, MATTR samples windows of size $W$ to calculate a stable mean richness score ([Covington & McFall, 2010](https://www.researchgate.net/publication/228913611_Cutting_the_Gordian_Knot_The_Moving-Average_Type-Token_Ratio)).
- **Syntactic Complexity**: Extracts the mean dependency parse depth. AI-generated prose often exhibits more uniform subordination patterns compared to human writing.
- **Semantic Flow**: Measures sequential coherence via the dot product of adjacent sentence embeddings. We assume that LLM prose displays 'hyper-coherence' – unnaturally smooth transitions compared to student drafts.

In [ ]:
# ---------------------------------------------------------------------------
# Each extractor is a small, self-contained callable dataclass.
# Adding a new one requires: implement extract(), register in ALL_EXTRACTORS.
# ---------------------------------------------------------------------------

@dataclass
class SurfaceExtractor:
  '''
  Character- and token-level surface statistics.
  These are fast, interpretable, and form a useful baseline.
  '''
  name: str = 'surface'

  def extract(self, text: str) -> dict[str, float]:
    tokens = tokenise_words(text)
    sents  = sent_tokenize(text)
    words  = len(tokens)

    sent_lens = [len(word_tokenize(s)) for s in sents]

    avg_word_len = (
      np.mean([len(t) for t in tokens]) if tokens else 0.0
    )

    return {
      'word_count':           float(words),
      'sentence_count':       float(len(sents)),
      'avg_sentence_length':  float(np.mean(sent_lens)) if sent_lens else 0.0,
      'std_sentence_length':  float(np.std(sent_lens))  if sent_lens else 0.0,
      'avg_word_length':      avg_word_len,
      # Char:word ratio -- inflated prose often has longer words
      'char_word_ratio':      float(len(text.replace(' ', '')) / max(words, 1)),
      'colon_per_100w':       text.count(':')  * 100 / max(words, 1),
      'semicolon_per_100w':   text.count(';')  * 100 / max(words, 1),
      'exclaim_per_100w':     text.count('!')  * 100 / max(words, 1),
      'question_per_100w':    text.count('?')  * 100 / max(words, 1),
      'comma_per_100w':       text.count(',')  * 100 / max(words, 1),
      'dash_per_100w':        (text.count('-') + text.count('\u2013') + text.count('\u2014'))
                              * 100 / max(words, 1),
      'bracket_per_100w':     (text.count('(') + text.count('['))
                              * 100 / max(words, 1),
    }


@dataclass
class LexicalExtractor:
  '''
  Vocabulary richness and lexical diversity features.
  TTR is sensitive to length; MATTR (moving-average TTR) corrects for that.
  '''
  name:        str = 'lexical'
  window_size: int = 50    # MATTR window

  def extract(self, text: str) -> dict[str, float]:
    tokens = tokenise_words(text)
    n      = len(tokens)
    if n == 0:
      return {k: 0.0 for k in [
        'ttr', 'mattr', 'hapax_ratio', 'top10_coverage',
        'stopword_ratio', 'content_word_ratio', 'long_word_ratio',
      ]}

    from nltk.corpus import stopwords as _sw
    stop = frozenset(_sw.words('english'))

    freq    = pd.Series(tokens).value_counts()
    hapax   = int((freq == 1).sum())
    top10   = freq.head(10).sum()

    # Moving-average TTR
    mattr_scores = [
      len(set(tokens[i:i + self.window_size])) / self.window_size
      for i in range(0, max(1, n - self.window_size + 1))
    ]

    content = [t for t in tokens if t not in stop and t.isalpha()]

    return {
      'ttr':              len(set(tokens)) / n,
      'mattr':            float(np.mean(mattr_scores)),
      'hapax_ratio':      hapax / n,
      'top10_coverage':   top10 / n,
      'stopword_ratio':   sum(1 for t in tokens if t in stop) / n,
      'content_word_ratio': len(content) / n,
      # Long words (>= 7 chars) correlate with formal / AI prose
      'long_word_ratio':  sum(1 for t in tokens if len(t) >= 7) / n,
    }


@dataclass
class SyntacticExtractor:
  '''
  POS-tag distribution features extracted via spaCy.
  The ratios of ADJ/ADV/NOUN/VERB signal different prose styles.
  '''
  name: str = 'syntactic'

  def extract(self, text: str) -> dict[str, float]:
    doc    = _NLP(text[:100_000])   # spaCy's default max is generous; clip for safety
    tags   = [t.pos_ for t in doc if not t.is_space]
    n      = max(len(tags), 1)
    counts = pd.Series(tags).value_counts()

    def ratio(pos: str) -> float:
      return counts.get(pos, 0) / n

    # Passive voice: simple heuristic (auxpass pattern no longer in spaCy v3;
    # we approximate it via 'be' aux + VERB with 'VBN' tag)
    passive_count = sum(
      1 for token in doc
      if token.dep_ == 'auxpass'
      or (token.lemma_ == 'be' and token.head.tag_ == 'VBN')
    )

    # Mean syntactic depth (a proxy for subordination complexity)
    depths = [len(list(token.ancestors)) for token in doc if not token.is_space]
    mean_depth = float(np.mean(depths)) if depths else 0.0

    return {
      'pos_noun_ratio':    ratio('NOUN'),
      'pos_verb_ratio':    ratio('VERB'),
      'pos_adj_ratio':     ratio('ADJ'),
      'pos_adv_ratio':     ratio('ADV'),
      'pos_pron_ratio':    ratio('PRON'),
      'pos_propn_ratio':   ratio('PROPN'),
      'pos_conj_ratio':    ratio('CCONJ') + ratio('SCONJ'),
      'passive_per_100w':  passive_count * 100 / n,
      'mean_parse_depth':  mean_depth,
      # Named-entity density -- AI essays often have more generic NEs
      'ner_density':       len(doc.ents) * 100 / n,
    }


@dataclass
class SemanticExtractor:
  '''
  Embedding-based semantic features: coherence, redundancy, and drift.
  All use SBERT embeddings for genuine semantic similarity.
  '''
  name: str = 'semantic'

  def extract(self, text: str) -> dict[str, float]:
    paras = split_paragraphs(text)
    sents = sent_tokenize(text)

    sent_vecs = embed_sentences(sents)
    para_vecs = embed_sentences(paras) if paras else np.zeros((0, 384))

    # -- Sequential sentence coherence (mean cosine similarity of adjacent pairs)
    seq_coh = self._sequential_similarity(sent_vecs)

    # -- Global semantic spread (mean pairwise distance -> diversity)
    spread = self._mean_pairwise_distance(sent_vecs)

    # -- Intro / outro similarity
    intro_outro = (
      float(para_vecs[0] @ para_vecs[-1])   # dot product; vecs are normalised
      if len(para_vecs) >= 2 else 0.0
    )

    # -- Intra-paragraph redundancy
    para_redundancies = self._intra_para_redundancy(paras)

    # -- Lexical vs semantic diversity gap:
    #    AI prose can be lexically rich but semantically repetitive
    lex_ttr  = LexicalExtractor().extract(text)['ttr']
    sem_div  = 1.0 - float(np.mean([
      float(sent_vecs[i] @ sent_vecs[j])
      for i in range(len(sent_vecs))
      for j in range(i + 1, len(sent_vecs))
    ])) if len(sent_vecs) > 1 else 0.0

    return {
      'sequential_coherence':       seq_coh,
      'semantic_spread':            spread,
      'intro_outro_similarity':     intro_outro,
      'mean_intra_para_redundancy': float(np.mean(para_redundancies)) if para_redundancies else 0.0,
      'max_intra_para_redundancy':  float(np.max(para_redundancies))  if para_redundancies else 0.0,
      # Positive gap -> lexically varied but semantically flat (AI signal)
      'lex_sem_diversity_gap':      lex_ttr - (1.0 - sem_div),
    }

  @staticmethod
  def _sequential_similarity(vecs: np.ndarray) -> float:
    if len(vecs) < 2:
      return 0.0
    return float(np.mean([vecs[i] @ vecs[i + 1] for i in range(len(vecs) - 1)]))

  @staticmethod
  def _mean_pairwise_distance(vecs: np.ndarray) -> float:
    if len(vecs) < 2:
      return 0.0
    sims = [
      float(vecs[i] @ vecs[j])
      for i in range(len(vecs))
      for j in range(i + 1, len(vecs))
    ]
    return 1.0 - float(np.mean(sims))

  @staticmethod
  def _intra_para_redundancy(paras: list[str]) -> list[float]:
    scores = []
    for para in paras:
      sents = sent_tokenize(para)
      if len(sents) < 2:
        continue
      vecs = embed_sentences(sents)
      sims = [
        float(vecs[i] @ vecs[j])
        for i in range(len(vecs))
        for j in range(i + 1, len(vecs))
      ]
      if sims:
        scores.append(float(np.mean(sims)))
    return scores


@dataclass
class StructuralExtractor:
  '''
  Essay-level structural signals: paragraph balance, signposting patterns,
  and the degree to which the essay follows a formulaic template.
  '''
  name: str = 'structural'

  # Common AI signpost openers (deliberately broader than the expert list)
  _OPENERS: frozenset[str] = frozenset({
    'furthermore', 'moreover', 'additionally', 'in conclusion',
    'in summary', 'firstly', 'secondly', 'thirdly', 'finally',
    'in addition', 'consequently', 'therefore', 'to conclude',
    'to summarise', 'it is worth noting', 'it is important',
    'in this essay', 'this essay will', 'as discussed',
  })

  def extract(self, text: str) -> dict[str, float]:
    paras  = split_paragraphs(text)
    tokens = tokenise_words(text)
    n_w    = max(len(tokens), 1)
    sents  = sent_tokenize(text)

    para_lens = [len(word_tokenize(p)) for p in paras]

    # Paragraph uniformity: low CV -> very uniform -> AI signal
    cv_paras = (
      float(np.std(para_lens) / np.mean(para_lens))
      if len(para_lens) > 1 and np.mean(para_lens) > 0
      else 0.0
    )

    opener_hits = sum(
      1 for s in sents
      if any(s.lower().strip().startswith(op) for op in self._OPENERS)
    )

    # 'Concession-assertion' pattern: 'While X, Y' / 'Although X, Y'
    concession_pattern = re.compile(
      r'\b(while|although|even though|whereas|despite)\b',
      re.IGNORECASE,
    )

    # First-person pronouns -- student voice marker
    first_person = re.compile(r'\b(i|me|my|myself|we|our|us)\b', re.IGNORECASE)

    return {
      'para_count':              float(len(paras)),
      'para_cv':                 cv_paras,                # coefficient of variation
      'opener_signpost_ratio':   opener_hits / max(len(sents), 1),
      'concession_per_100w':     len(concession_pattern.findall(text)) * 100 / n_w,
      'first_person_per_100w':   len(first_person.findall(text)) * 100 / n_w,
      # Sentence-initial capitalised noun (a rough template-conformity check)
      'para_uniformity':         1.0 - cv_paras,          # inverted so higher -> more uniform
    }


# ---------------------------------------------------------------------------
# Registry -- add new extractors here; the rest of the pipeline picks them up.
# ---------------------------------------------------------------------------
ALL_EXTRACTORS: list[FeatureExtractor] = [
  SurfaceExtractor(),
  LexicalExtractor(),
  SyntacticExtractor(),
  SemanticExtractor(),
  StructuralExtractor(),
]

## Feature Extraction Pipeline

This section constructs the `FeatureMatrix` by iterating through the `LabelledCorpus`.

### Implementation
- **Fault Tolerance**: The pipeline catches extractor exceptions per essay, ensuring a single malformed string does not terminate the batch process.
- **Namespacing**: Features are prefixed with their extractor name (e.g., `lexical__ttr`) to maintain provenance and prevent collision.

In [ ]:
def extract_features(text: str, extractors: list[FeatureExtractor] | None = None) -> dict[str, float]:
  '''
  Run all registered extractors over *text* and merge their outputs.
  Extractor names are prepended to feature keys to avoid collisions
  (e.g. 'semantic__sequential_coherence').
  '''
  # Default to the global registry if no specific list of extractors is provided.
  extractors = extractors or ALL_EXTRACTORS
  features: dict[str, float] = {}
  # Execute each extractor and prefix the keys to ensure uniqueness in the feature set.
  for extractor in extractors:
    raw = extractor.extract(text)
    features.update({f'{extractor.name}__{k}': v for k, v in raw.items()})
  return features


def build_feature_matrix(
  corpus: LabelledCorpus,
  extractors: list[FeatureExtractor] | None = None,
) -> FeatureMatrix:
  '''
  Extract features for every essay in the labelled corpus.
  Returns a DataFrame with one row per essay and a 'label' column.

  Parameters
  ----------
  corpus:
      Dict mapping group label to list of essay strings.
  extractors:
      Which extractors to use; defaults to ALL_EXTRACTORS.

  Notes
  -----
  Extractor failures are caught per-essay and filled with NaN so that
  one bad document does not abort the whole run.
  '''
  rows: list[dict[str, Any]] = []
  # Iterate through categories and their respective essay lists.
  for label, essays in corpus.items():
    for i, essay in enumerate(essays):
      try:
        # Attempt to derive the linguistic features.
        feats = extract_features(essay, extractors)
      except Exception as exc:  # noqa: BLE001
        # Log a warning and proceed with an empty dictionary if extraction fails.
        print(f'  [warn] extractor failed on {label}[{i}]: {exc}')
        feats = {}
      # Append metadata along with the extracted feature values.
      rows.append({'label': label, 'essay_index': i, **feats})

  # Construct the final DataFrame.
  df = pd.DataFrame(rows)
  # Ensure numeric columns are float
  meta_cols = {'label', 'essay_index'}
  # Filter out metadata to target specific feature columns for type casting.
  num_cols  = [c for c in df.columns if c not in meta_cols]
  df[num_cols] = df[num_cols].astype(float)
  return df

## Anomaly Detection (Unsupervised)

We utilise an Isolation Forest to identify essays that are statistically atypical within the high-dimensional feature space.

### Theoretical Framework
- **Isolation Forest**: This algorithm isolates observations by randomly selecting a feature and a split value. The anomaly score is derived from the average path length $h(x)$ in a forest of $iTrees$: $s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$. Outliers have significantly shorter paths ([Liu et al., 2008](https://ieeexplore.ieee.org/document/4781136)).
- **Assumption**: We assume that early generative AI responses or extreme human outliers will manifest as low-density points in the feature manifold.

In [ ]:
# ---------------------------------------------------------------------------
# Unsupervised stage: no labels needed.
# IsolationForest flags essays that are anomalous in feature-space.
# Useful as a pre-screening step before labelling.
# ---------------------------------------------------------------------------

def detect_anomalies(
  feature_df: FeatureMatrix,
  *,
  contamination: float = 0.15,
  random_state:  int   = 42,
) -> pd.Series:
  '''
  Fit an IsolationForest and return anomaly scores (lower -> more anomalous).
  The score is negated so that larger values indicate more 'normal' essays.

  Parameters
  ----------
  contamination:
      Expected fraction of outliers; 0.15 is a conservative default.
  '''
  num_cols = feature_df.select_dtypes(include='number').columns.tolist()
  X        = feature_df[num_cols].fillna(0).values

  scaler = StandardScaler()
  X_sc   = scaler.fit_transform(X)

  iso = IsolationForest(contamination=contamination, random_state=random_state, n_jobs=-1)
  iso.fit(X_sc)

  scores = iso.decision_function(X_sc)   # higher -> more normal
  return pd.Series(scores, index=feature_df.index, name='anomaly_score')

## Supervised Feature Importance (Random Forest)

A Random Forest classifier is trained to discriminate between pre- and post-AI cohorts, identifying the most predictive linguistic markers.

### Theoretical Framework
- **Ensemble Learning**: Random Forests mitigate overfitting by averaging uncorrelated trees. Feature importance is calculated via Gini Impurity: $G = 1 - \sum_{i=1}^{c} (p_i)^2$.
- **SHAP Values**: Based on Shapley values from cooperative game theory, SHAP provides a consistent additive feature attribution: $\phi_i = \sum_{S \subseteq \{x_1, \dots, x_p\} \setminus \{x_i\}} \frac{|S|!(p-|S|-1)!}{p!} (f(S \cup \{x_i\}) - f(S))$. This reveals the *direction* of influence (e.g., whether higher punctuation density pushes a prediction toward 'Post-ChatGPT') ([Lundberg & Lee, 2017](https://arxiv.org/abs/1705.07874)).

In [ ]:
def rank_features_by_importance(
  feature_df: FeatureMatrix,
  label_col:  str = 'label',
  *,
  n_estimators: int = 300,
  random_state: int = 42,
) -> pd.DataFrame:
  '''
  Train a Random Forest classifier on the feature matrix and return a
  DataFrame of features ranked by Gini importance.

  The Random Forest is chosen here because:
  - it handles correlated features gracefully (unlike LASSO),
  - it gives intrinsic feature importances without a separate SHAP call,
  - it requires minimal hyperparameter tuning for a first-pass analysis.

  Parameters
  ----------
  feature_df:
      Output of build_feature_matrix().
  label_col:
      Column name containing group labels.

  Returns
  -------
  DataFrame with columns ['feature', 'importance', 'std'] sorted descending.
  '''
  meta_cols = {label_col, 'essay_index'}
  num_cols  = [c for c in feature_df.columns if c not in meta_cols]

  X = feature_df[num_cols].fillna(0).values
  y = feature_df[label_col].values

  scaler = StandardScaler()
  X_sc   = scaler.fit_transform(X)

  rf = RandomForestClassifier(
    n_estimators=n_estimators,
    max_features='sqrt',
    random_state=random_state,
    n_jobs=-1,
  )
  rf.fit(X_sc, y)

  importance_df = pd.DataFrame({
    'feature':    num_cols,
    'importance': rf.feature_importances_,
    'std':        np.std(
      [tree.feature_importances_ for tree in rf.estimators_], axis=0
    ),
  }).sort_values('importance', ascending=False).reset_index(drop=True)

  return importance_df, rf, scaler, num_cols


def compute_shap_values(
  rf: RandomForestClassifier,
  X_sc: np.ndarray,
  feature_names: list[str],
  max_display: int = 15,
) -> shap.Explanation:
  '''
  Compute SHAP TreeExplainer values for the fitted Random Forest.
  Returns a shap.Explanation object for flexible downstream use.

  SHAP is preferred over raw Gini importance because it is:
  - unbiased toward high-cardinality features,
  - consistent (a feature always gets 0 SHAP if it does not help),
  - directional (sign tells you which group a feature pushes toward).
  '''
  explainer = shap.TreeExplainer(rf)
  shap_vals  = explainer(X_sc)
  return shap_vals

## Metric Suggestion Engine

This engine synthesises statistical importance into interpretable pedagogical insights.

### Logic
- **Directionality**: By comparing the mean vector $\boldsymbol{\mu}_{pre}$ against $\boldsymbol{\mu}_{post}$, the engine classifies features as markers of AI adoption or markers of human voice (student presence).
- **Heuristics**: We assume features with an importance score $> 0.01$ are sufficiently stable to be considered candidate metrics for broader class-level screening.

In [ ]:
def suggest_metrics(
  feature_df:    FeatureMatrix,
  importance_df: pd.DataFrame,
  label_col:     str   = 'label',
  top_n:         int   = 10,
  *,
  min_importance: float = 0.01,
) -> list[DiscoveredMetric]:
  '''
  Translate the ranked feature importance table into human-readable
  DiscoveredMetric suggestions.

  Direction is determined by comparing per-group means: whichever group
  has the higher mean for a feature gets credit for that direction.

  Parameters
  ----------
  top_n:
      How many metrics to surface.
  min_importance:
      Features below this Gini importance are silently dropped.
  '''
  labels    = feature_df[label_col].unique().tolist()
  top_feats = importance_df[importance_df['importance'] >= min_importance].head(top_n)

  suggestions: list[DiscoveredMetric] = []
  for _, row in top_feats.iterrows():
    feat = row['feature']
    imp  = float(row['importance'])

    # Per-group means for direction inference
    group_means: dict[str, float] = {
      lbl: float(feature_df.loc[feature_df[label_col] == lbl, feat].mean())
      for lbl in labels
    }

    if len(labels) == 2:
      l1, l2 = labels
      direction = (
        f'higher_in_{_slug(l1)}' if group_means[l1] > group_means[l2]
        else f'higher_in_{_slug(l2)}'
      )
    else:
      direction = 'unclear'

    suggestions.append(DiscoveredMetric(
      name          = feat,
      importance    = imp,
      direction     = direction,
      description   = _describe_feature(feat),
      example_values = group_means,
    ))

  return suggestions


def _slug(s: str) -> str:
  '''Turn a label into a safe identifier fragment.'''
  return re.sub(r'\W+', '_', s).strip('_').lower()


# ---------------------------------------------------------------------------
# Human-readable descriptions for known feature name patterns.
# Extend this dict as new extractors are added.
# ---------------------------------------------------------------------------
_FEATURE_DESCRIPTIONS: dict[str, str] = {
  'surface__word_count':           'Total word count of the essay.',
  'surface__avg_sentence_length':  'Mean number of words per sentence.',
  'surface__std_sentence_length':  'Variability in sentence length (burstiness).',
  'surface__colon_per_100w':       'Colon frequency per 100 words.',
  'surface__semicolon_per_100w':   'Semicolon frequency per 100 words.',
  'surface__comma_per_100w':       'Comma frequency per 100 words.',
  'surface__dash_per_100w':        'Dash frequency per 100 words.',
  'surface__avg_word_length':      'Mean character count per word.',
  'lexical__ttr':                  'Type-token ratio (raw vocabulary richness).',
  'lexical__mattr':                'Moving-average TTR (length-corrected richness).',
  'lexical__hapax_ratio':          'Fraction of words appearing only once.',
  'lexical__long_word_ratio':      'Fraction of words >= 7 characters.',
  'lexical__stopword_ratio':       'Fraction of tokens that are stopwords.',
  'lexical__content_word_ratio':   'Fraction of tokens that are content words.',
  'syntactic__pos_adj_ratio':      'Proportion of adjectives in the text.',
  'syntactic__pos_adv_ratio':      'Proportion of adverbs.',
  'syntactic__pos_noun_ratio':     'Proportion of nouns.',
  'syntactic__pos_verb_ratio':     'Proportion of verbs.',
  'syntactic__pos_pron_ratio':     'Proportion of pronouns (first-person signal).',
  'syntactic__passive_per_100w':   'Passive voice constructions per 100 words.',
  'syntactic__mean_parse_depth':   'Mean syntactic parse depth (subordination complexity).',
  'syntactic__ner_density':        'Named entity density per 100 tokens.',
  'semantic__sequential_coherence':'Mean cosine similarity between adjacent sentences (flow).',
  'semantic__semantic_spread':     'Mean pairwise sentence distance (topical diversity).',
  'semantic__intro_outro_similarity': 'Semantic similarity between first and last paragraph.',
  'semantic__mean_intra_para_redundancy': 'Average semantic redundancy within paragraphs.',
  'semantic__lex_sem_diversity_gap': 'Gap between lexical and semantic diversity (AI signal).',
  'structural__para_uniformity':   'How uniform paragraph lengths are (high -> template-like).',
  'structural__opener_signpost_ratio': 'Fraction of sentences starting with discourse markers.',
  'structural__first_person_per_100w': 'First-person pronoun rate (student voice).',
  'structural__concession_per_100w': 'Concessive constructions per 100 words.',
}


def _describe_feature(feature_name: str) -> str:
  '''Return a human-readable description or a sensible fallback.'''
  if feature_name in _FEATURE_DESCRIPTIONS:
    return _FEATURE_DESCRIPTIONS[feature_name]
  # Fallback: prettify the name
  _, _, short = feature_name.partition('__')
  return short.replace('_', ' ').capitalize() + '.'

## Dimensionality Reduction & Embedding Visualisation

We project the high-dimensional feature space into a 2D plane to visualise the separation between student and AI-augmented cohorts.

### Techniques
- **UMAP**: A manifold learning technique based on topological data analysis. It preserves both local and global structure by optimising the cross-entropy between fuzzy simplicial sets ([McInnes et al., 2018](https://arxiv.org/abs/1802.03426)).
- **PCA**: A linear orthogonal transformation used here for deterministic baseline analysis when sample sizes are small ($n < 20$). It identifies the axes of maximum variance in the data.

In [ ]:
def reduce_dimensions(
  feature_df: FeatureMatrix,
  method:     str = 'umap',
  *,
  n_components: int = 2,
  random_state: int = 42,
) -> np.ndarray:
  '''
  Project the high-dimensional feature matrix to 2D for visual inspection.

  Supported methods:
  - 'umap'  -- preserves local + global structure; best for clustering insight.
  - 'tsne'  -- preserves local structure; can be slow for large n.
  - 'pca'   -- linear; fastest; useful sanity check.

  UMAP is the default because it scales well and yields interpretable
  neighbourhood structure even for small corpora.
  '''
  meta_cols = {'label', 'essay_index'}
  num_cols  = [c for c in feature_df.columns if c not in meta_cols]
  X         = feature_df[num_cols].fillna(0).values

  scaler = StandardScaler()
  X_sc   = scaler.fit_transform(X)

  match method:
    case 'umap':
      reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=min(15, len(X_sc) - 1),
        min_dist=0.1,
        random_state=random_state,
      )
    case 'tsne':
      reducer = TSNE(
        n_components=n_components,
        perplexity=min(30, len(X_sc) - 1),
        random_state=random_state,
      )
    case 'pca':
      reducer = PCA(n_components=n_components, random_state=random_state)
    case _:
      raise ValueError(f'Unknown method "{method}". Choose "umap", "tsne", or "pca".')

  return reducer.fit_transform(X_sc)

## Visualisation

In [ ]:
# ---------------------------------------------------------------------------
# All plots use the Agrarian palette and are designed to be readable in print
# (no colour-only encoding; markers and hatching are used where possible).
# ---------------------------------------------------------------------------

_LABEL_COLOURS: dict[str, str] = {}   # populated lazily in _colour_for()


def _colour_for(label: str, palette: list[str] = _PALETTE_FOUR) -> str:
  '''Assign a stable colour to each label on first use.'''
  if label not in _LABEL_COLOURS:
    _LABEL_COLOURS[label] = palette[len(_LABEL_COLOURS) % len(palette)]
  return _LABEL_COLOURS[label]


def plot_feature_importance(
  importance_df: pd.DataFrame,
  top_n:         int   = 20,
  figsize:       tuple[float, float] = (10, 7),
) -> plt.Figure:
  '''
  Horizontal bar chart of the top-N most discriminative features,
  with error bars showing variability across trees.
  '''
  df    = importance_df.head(top_n).copy()
  df    = df.sort_values('importance')     # ascending so top lands at the top
  labels = [_prettify(f) for f in df['feature']]

  fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
  bars = ax.barh(
    labels, df['importance'],
    xerr=df['std'],
    color=UAF['blue'], ecolor=UAF['sky'],
    capsize=3, alpha=0.9,
  )
  ax.set_xlabel('Gini Importance')
  ax.set_title(f'Top {top_n} Discriminative Features (Random Forest)', fontweight='bold')
  # Accessible: add value annotations
  for bar, val in zip(bars, df['importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=8)
  return fig


def plot_embedding(
  feature_df: FeatureMatrix,
  coords:     np.ndarray,
  label_col:  str   = 'label',
  method:     str   = 'UMAP',
  figsize:    tuple[float, float] = (8, 6),
) -> plt.Figure:
  '''
  Scatter plot of dimensionally-reduced essay representations.
  Each group gets a distinct colour AND marker for print accessibility.
  '''
  labels  = feature_df[label_col].values
  markers = ['o', 's', '^', 'D', 'v', 'P']   # distinct shapes

  fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
  for i, lbl in enumerate(pd.unique(labels)):
    mask = labels == lbl
    ax.scatter(
      coords[mask, 0], coords[mask, 1],
      label=lbl,
      color=_colour_for(lbl),
      marker=markers[i % len(markers)],
      s=80, alpha=0.85, edgecolors='white', linewidths=0.5,
    )
  ax.set_xlabel(f'{method} 1')
  ax.set_ylabel(f'{method} 2')
  ax.set_title(f'Essay Embedding Space ({method})', fontweight='bold')
  ax.legend(framealpha=0.9)
  return fig


def plot_metric_suggestions(
  suggestions:   list[DiscoveredMetric],
  label_col:     str   = 'label',
  figsize:       tuple[float, float] = (12, 7),
) -> plt.Figure:
  '''
  Grouped bar chart showing per-group mean values for each suggested metric,
  alongside its importance score.
  '''
  if not suggestions:
    fig, ax = plt.subplots()
    ax.text(0.5, 0.5, 'No suggestions to display.', ha='center', va='center')
    return fig

  n     = len(suggestions)
  names = [_prettify(s.name) for s in suggestions]
  all_labels = list(suggestions[0].example_values.keys())
  n_groups   = len(all_labels)
  width      = 0.8 / n_groups
  x          = np.arange(n)

  fig, axes = plt.subplots(2, 1, figsize=figsize, constrained_layout=True,
                            gridspec_kw={'height_ratios': [3, 1]})

  # -- Top panel: grouped bars of per-group means
  ax = axes[0]
  for gi, lbl in enumerate(all_labels):
    vals   = [s.example_values.get(lbl, 0.0) for s in suggestions]
    offset = (gi - (n_groups - 1) / 2) * width
    bars   = ax.bar(x + offset, vals, width, label=lbl,
                    color=_colour_for(lbl), alpha=0.9)

  ax.set_xticks(x)
  ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
  ax.set_ylabel('Mean feature value')
  ax.set_title('Suggested Metrics: Per-Group Means', fontweight='bold')
  ax.legend()

  # -- Bottom panel: importance scores
  ax2 = axes[1]
  ax2.bar(x, [s.importance for s in suggestions],
          color=UAF['yellow'], edgecolor=UAF['blue'], linewidth=0.8)
  ax2.set_xticks(x)
  ax2.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
  ax2.set_ylabel('Importance')
  ax2.set_title('Feature Importance', fontweight='bold')

  return fig


def plot_shap_summary(
  shap_vals:     shap.Explanation,
  feature_names: list[str],
  max_display:   int = 15,
) -> plt.Figure:
  '''
  SHAP beeswarm plot (matplotlib backend) showing feature impact direction.
  Each dot is one essay; colour encodes feature value.
  '''
  fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
  # shap's plot functions draw onto the current axes
  shap.plots.beeswarm(
    shap_vals[:, :, 1] if shap_vals.values.ndim == 3 else shap_vals,
    max_display=max_display,
    show=False,
    color_bar=True,
    plot_size=None,
  )
  plt.title('SHAP Feature Impact (Post-ChatGPT direction)', fontweight='bold')
  return fig


def plot_anomaly_scores(
  feature_df:     FeatureMatrix,
  anomaly_scores: pd.Series,
  label_col:      str   = 'label',
  figsize:        tuple[float, float] = (9, 5),
) -> plt.Figure:
  '''
  Strip + box plot of anomaly scores per group.
  Low scores flag essays that are unusual within their group.
  '''
  plot_df = feature_df[[label_col]].copy()
  plot_df['anomaly_score'] = anomaly_scores.values

  fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
  order = plot_df[label_col].unique().tolist()
  sns.boxplot(data=plot_df, x=label_col, y='anomaly_score', ax=ax,
              order=order, palette={lbl: _colour_for(lbl) for lbl in order},
              width=0.4, fliersize=0)
  sns.stripplot(data=plot_df, x=label_col, y='anomaly_score', ax=ax,
                order=order, color='black', alpha=0.5, size=5, jitter=True)
  ax.axhline(0, linestyle='--', color='grey', linewidth=0.8)
  ax.set_ylabel('Anomaly Score (higher = more typical)')
  ax.set_xlabel('')
  ax.set_title('Essay Anomaly Scores by Group', fontweight='bold')
  return fig


def _prettify(feature_name: str) -> str:
  '''Strip extractor prefix and humanise underscores for axis labels.'''
  _, _, short = feature_name.partition('__')
  return short.replace('_', ' ').title()

## Metric Report

In [ ]:
def print_metric_report(suggestions: list[DiscoveredMetric]) -> None:
  '''
  Print a structured plain-text report of discovered metrics.
  Intentionally plain so it can be redirected to a file or pasted into a doc.
  '''
  sep = '=' * 64
  print(sep)
  print('ML-DISCOVERED METRIC SUGGESTIONS')
  print(sep)
  for rank, metric in enumerate(suggestions, start=1):
    print(f'\n#{rank:>2}  {metric.name}')
    print(f'     Importance : {metric.importance:.4f}')
    print(f'     Direction  : {metric.direction}')
    print(f'     Description: {metric.description}')
    if metric.example_values:
      for lbl, val in metric.example_values.items():
        print(f'     [{lbl}]: {val:.4f}')
  print(f'\n{sep}\n')

## Full Pipeline Orchestrator

In [ ]:
def run_ml_discovery(
  corpus:        LabelledCorpus,
  *,
  top_n:         int   = 10,
  reduction:     str   = 'umap',
  show_shap:     bool  = True,
  random_state:  int   = 42,
) -> dict[str, Any]:
  '''
  End-to-end ML metric discovery pipeline.

  Stages
  ------
  1. Feature extraction -- all registered extractors, all essays.
  2. Anomaly detection  -- flag unusual essays before supervised step.
  3. Supervised ranking -- Random Forest + SHAP to surface discriminative features.
  4. Metric suggestion  -- translate importance table to DiscoveredMetric objects.
  5. Visualisation      -- embedding space, importance chart, SHAP, anomaly scores.

  Parameters
  ----------
  corpus:
      {label: [essay_text, ...]} for each group (e.g. pre / post ChatGPT).
  top_n:
      Number of metrics to surface.
  reduction:
      Dimensionality reduction method for embedding plot ('umap', 'tsne', 'pca').
  show_shap:
      Whether to render the SHAP beeswarm (slower; skip for quick runs).

  Returns
  -------
  Dict containing all intermediate artefacts for downstream use.
  '''
  print('[1/5] Extracting features ...')
  feature_df = build_feature_matrix(corpus)
  print(f'      {len(feature_df)} essays x {len(feature_df.columns) - 2} features')

  print('[2/5] Detecting anomalies ...')
  anomaly_scores = detect_anomalies(feature_df, random_state=random_state)
  fig_anom = plot_anomaly_scores(feature_df, anomaly_scores)
  plt.show()

  print('[3/5] Ranking features (Random Forest) ...')
  importance_df, rf, scaler, num_cols = rank_features_by_importance(
    feature_df, random_state=random_state
  )
  fig_imp = plot_feature_importance(importance_df, top_n=min(20, len(importance_df)))
  plt.show()

  shap_vals = None
  if show_shap and len(feature_df) >= 4:    # SHAP needs at least a few samples
    print('[3b]  Computing SHAP values ...')
    X_sc      = scaler.transform(feature_df[num_cols].fillna(0).values)
    shap_vals = compute_shap_values(rf, X_sc, num_cols)
    fig_shap  = plot_shap_summary(shap_vals, num_cols)
    plt.show()

  print('[4/5] Generating metric suggestions ...')
  suggestions = suggest_metrics(feature_df, importance_df, top_n=top_n)
  print_metric_report(suggestions)
  fig_sugg = plot_metric_suggestions(suggestions)
  plt.show()

  print('[5/5] Projecting to embedding space ...')
  coords    = reduce_dimensions(feature_df, method=reduction, random_state=random_state)
  fig_embed = plot_embedding(feature_df, coords, method=reduction.upper())
  plt.show()

  print('Done.')
  return {
    'feature_df':    feature_df,
    'anomaly_scores': anomaly_scores,
    'importance_df': importance_df,
    'suggestions':   suggestions,
    'shap_vals':     shap_vals,
    'coords':        coords,
    'rf':            rf,
  }

## Demo

In [ ]:
# ---------------------------------------------------------------------------
# Demo using the same toy essays from 01_expert_metrics.ipynb.
# Replace these with real essay lists when your dataset arrives.
# ---------------------------------------------------------------------------

_PRE_AI_ESSAY = '''
The French Revolution fundamentally changed European politics. People were angry
about inequality and bread prices. The king didn't really understand what was
happening until it was too late. There were a lot of different causes - economic,
social, and political - but they all kind of built up together.

Robespierre is an interesting figure because he seemed to believe in what he was
doing even when it got violent. I think that's what makes him different from
someone who was just power-hungry.

In the end, the Revolution opened the door for Napoleon. Whether that was a good
thing or not is still debated.
'''

_POST_AI_ESSAY = '''
The French Revolution was a pivotal moment in European history that fundamentally
transformed the political landscape of the continent. It is important to note that
this transformative event was multifaceted in its causes; economic hardship,
social inequality, and political instability all played crucial roles.

Furthermore, the Revolution fostered a new paradigm of governance that would
underscore subsequent democratic movements. Moreover, figures such as Robespierre
navigated the complex intersection of idealism and pragmatism, leaving a
comprehensive legacy that continues to resonate.

In conclusion, the French Revolution was not merely a national event but a
testament to the broader human struggle for liberty. Its vibrant tapestry of
causes and consequences continues to shape our understanding of modern democracy.
'''

# Simulate small corpora -- replace with real data
# _CORPUS: LabelledCorpus = {
#   'Pre-ChatGPT':  [_PRE_AI_ESSAY]  * 4,
#   'Post-ChatGPT': [_POST_AI_ESSAY] * 4,
# }

# results = run_ml_discovery(
#   _CORPUS,
#   top_n=10,
#   reduction='pca',    # PCA is stable with n=8; switch to 'umap' with real data
#   show_shap=True,
# )

# For running with data ingestion
results = run_ml_discovery(CORPUS, top_n=10, reduction='umap')

## Design Notes

**Why these ML choices?**

| Choice | Rationale |
|---|---|
| SBERT `all-MiniLM-L6-v2` | Semantic similarity without an API key; free, offline after first download; 384-d embeddings are compact |
| Random Forest + SHAP | RF handles correlated features and small *n* well; SHAP gives directional importance that Gini alone cannot |
| IsolationForest | Unsupervised anomaly detection requires no labels -- useful for screening before you have clean corpora |
| UMAP over t-SNE | Preserves global structure; deterministic with `random_state`; faster |
| Protocol over ABC | Structural typing keeps extractors independent; adding one never touches existing code |

**Extending the pipeline**

- **New extractor**: implement `.name` and `.extract(text) -> dict[str, float]`; append to `ALL_EXTRACTORS`.
- **New file format** (PDF, DOCX, etc.): add a loader that returns `str` and pass it to `build_feature_matrix` -- no other changes needed.
- **Larger corpora**: the bottleneck is SBERT; consider batching via `SentenceTransformer.encode(..., batch_size=64)`.
- **Multi-class** (e.g. per-semester): `rank_features_by_importance` and `suggest_metrics` already support `n > 2` labels.